In [19]:
import sys
from pathlib import Path
import os

sys.path.append(str(Path().resolve().parents[2]))

In [20]:
from app.core import md5
from app.core.lab1 import LCG

In [21]:
BLOCK_SIZE = 8
W = 32   # довжина слова
R = 20   # кількість раундів
B = 32   # довжина ключа в байтах
P = 0xB7E15163
Q = 0x9E3779B9
S = []

In [22]:
lcg = LCG()
iv = lcg.generate_iv()
print(iv)

b'm\x8a\x01\x00m\xea\xa6\x18'


In [23]:
def rot_left(x: int, y: int):
    y %= W
    return ((x << y) & 0xFFFFFFFF) | (x >> (W - y))

In [24]:
def rot_right(x: int, y: int) -> int:
    y %= W
    return (x >> y) | ((x << (W - y)) & 0xFFFFFFFF)

In [25]:
def pad(data: bytes):
    pad_len = BLOCK_SIZE - (len(data) % BLOCK_SIZE)
    return data + bytes([pad_len] * pad_len)

In [26]:
def unpad(data: bytes):
    pad_len = data[-1]
    return data[:-pad_len]

In [27]:
def key_expansion(key_bytes: bytes):
    global S
    C = (B * 8 + W - 1) // W
    L = [0] * C
    for i in reversed(range(len(key_bytes))):
        L[i // 4] = (L[i // 4] << 8) + key_bytes[i]

    S = [0] * (2 * (R + 1))
    S[0] = P
    for i in range(1, len(S)):
        S[i] = (S[i - 1] + Q) & 0xFFFFFFFF

    n = 3 * max(C, len(S))
    A = Bv = i = j = 0
    for _ in range(n):
        A = S[i] = rot_left((S[i] + A + Bv) & 0xFFFFFFFF, 3)
        Bv = L[j] = rot_left((L[j] + A + Bv) & 0xFFFFFFFF, (A + Bv) & 0x1F)
        i = (i + 1) % len(S)
        j = (j + 1) % C

In [28]:
def encrypt(A: int, B: int):
    A = (A + S[0]) & 0xFFFFFFFF
    B = (B + S[1]) & 0xFFFFFFFF
    for i in range(1, R + 1):
        A = (rot_left(A ^ B, B) + S[2 * i]) & 0xFFFFFFFF
        B = (rot_left(B ^ A, A) + S[2 * i + 1]) & 0xFFFFFFFF
    return A, B

In [29]:
def decrypt(A: int, B: int):
    for i in range(R, 0, -1):
        B = rot_right((B - S[2 * i + 1]) & 0xFFFFFFFF, A) ^ A
        A = rot_right((A - S[2 * i]) & 0xFFFFFFFF, B) ^ B
    B = (B - S[1]) & 0xFFFFFFFF
    A = (A - S[0]) & 0xFFFFFFFF
    return A, B

In [30]:
def encrypt_block(block: bytes):
    A = int.from_bytes(block[:4], "little")
    B = int.from_bytes(block[4:], "little")

    A_enc, B_enc = encrypt(A, B)
    return A_enc.to_bytes(4, "little") + B_enc.to_bytes(4, "little")

In [31]:
def decrypt_block(block: bytes):
    A = int.from_bytes(block[:4], "little")
    B = int.from_bytes(block[4:], "little")

    A_dec, B_dec = decrypt(A, B)
    return A_dec.to_bytes(4, "little") + B_dec.to_bytes(4, "little")

In [32]:
def encrypt_cbc(data: bytes, iv: bytes):
    data = pad(data)
    result = b""
    prev = iv

    for i in range(0, len(data), BLOCK_SIZE):
        block = data[i:i+BLOCK_SIZE]

        xored = bytes(a ^ b for a, b in zip(block, prev))

        cipher_block = encrypt_block(xored)

        result += cipher_block
        prev = cipher_block

    return result

In [33]:
def decrypt_cbc(data: bytes, iv: bytes):
    result = b""
    prev = iv

    for i in range(0, len(data), BLOCK_SIZE):
        block = data[i:i+BLOCK_SIZE]

        decrypted = decrypt_block(block)

        # XOR назад
        plain_block = bytes(a ^ b for a, b in zip(decrypted, prev))

        result += plain_block
        prev = block

    return unpad(result)

In [34]:
pwd = "екр"

In [35]:
m1 = md5.MD5()
m1.update(pwd.encode())
h1 = bytes.fromhex(m1.finalize())

m2 = md5.MD5()
m2.update(h1)
h2 = bytes.fromhex(m2.finalize())

key = h2 + h1
key_expansion(key)

In [36]:
encrypted_iv = encrypt_block(iv)

cipher = encrypt_cbc(b"hello world", iv)

final_cipher = encrypted_iv + cipher
print("CIPHER:", final_cipher)

CIPHER: b"\x96orG)\xc5&\x06k\xda'\xf6\x99\x9eW\xfe\x81\x17\xda\xb0\xb3U\xa3\x89"


In [37]:
enc_iv = final_cipher[:8]
cipher_data = final_cipher[8:]

iv = decrypt_block(enc_iv)

plain = decrypt_cbc(cipher_data, iv)

print("PLAIN:", plain)

PLAIN: b'hello world'
